## MASTER ACTIVATOR TEMPLATE

In [ ]:
import sys, os

_stale = ['chromadb','gradio','sentence_transformers', 'pydantic',
          'huggingface_hub','langchain','transformers', 'albumentations']
for m in list(sys.modules.keys()):
    if any(s in m for s in _stale):
        del sys.modules[m]

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# All project paths
LIB_DIR  = "/content/drive/MyDrive/radiology_ai/libs"
DATA_DIR = "/content/drive/MyDrive/radiology_ai/data"
MDL_DIR  = "/content/drive/MyDrive/radiology_ai/models"
RES_DIR  = "/content/drive/MyDrive/radiology_ai/results"
RAG_DIR  = "/content/drive/MyDrive/radiology_ai/rag_papers"
CHR_DIR  = "/content/drive/MyDrive/radiology_ai/chroma_db"

for d in [LIB_DIR,DATA_DIR,MDL_DIR,RES_DIR,RAG_DIR,CHR_DIR]:
    os.makedirs(d, exist_ok=True)

if LIB_DIR in sys.path: sys.path.remove(LIB_DIR)
sys.path.insert(0, LIB_DIR)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Libs loaded | Device: {DEVICE.upper()}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("No GPU — go to Runtime → Change runtime type → T4 GPU")

## Secure API Key Provisioning & Kaggle Setup

In [ ]:
import os
import json

# ENTER YOUR KAGGLE API TOKENS HERE
KAGGLE_USERNAME = " "   # ← Replace with your username
KAGGLE_TOKEN    = " "   # ← Replace with your token (key value from kaggle.json)

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

with open(f"{kaggle_dir}/kaggle.json", "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_TOKEN}, f)

os.chmod(f"{kaggle_dir}/kaggle.json", 0o600)
print("Kaggle Identity Credentials Synced Successfully!")

## Automated Dataset Download & Extraction

In [ ]:
import glob

NIH_DIR = f"{DATA_DIR}/nih"
os.makedirs(NIH_DIR, exist_ok=True)
print(f"Storage Destination Target: {NIH_DIR}")

print("Fetching NIH Chest X-Ray sample payload (~5GB)...")
!kaggle datasets download -d nih-chest-xrays/sample -p {NIH_DIR} --unzip

print("\n Evaluating structural output on disk...")
sample_imgs = glob.glob(f"{NIH_DIR}/**/*.png", recursive=True)

if sample_imgs:
    print(f"Download Verified! Found {len(sample_imgs):,} medical frames on disk.")
    csv_check = glob.glob(f"{NIH_DIR}/*.csv")
    if csv_check:
        print(f"Metadata Catalog Node Bound: {os.path.basename(csv_check[0])}")
else:
    print("Directory alignment mismatch. Reviewing directory contents:")
    !ls -R {NIH_DIR}

# Cleanup redundant archive residuals
zips = glob.glob(f"{NIH_DIR}/*.zip")
for z in zips:
    os.remove(z)
print("Workspace purged of compressed installation archives.")

## EDA + Create Splits

In [ ]:
import pandas as pd
import os

# Update path to where the sample unzipped
csv_path = f"{NIH_DIR}/sample_labels.csv"

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("Dataset loaded successfully!")
    print(f"Total images for training/testing: {len(df)}")
    print("\nLabel Distribution:")
    # Split the multi-labels and count them
    print(df['Finding Labels'].str.get_dummies(sep='|').sum().sort_values(ascending=False))
else:
    print("Metadata CSV not found. Check the folder structure using !ls -R")

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib, json, os, glob
from sklearn.model_selection import train_test_split

# --- Using your exact established paths ---
NIH_DIR = "/content/drive/MyDrive/radiology_ai/data/nih"
DATA_DIR = "/content/drive/MyDrive/radiology_ai/data"
RES_DIR = "/content/drive/MyDrive/radiology_ai/results"
IMG_DIR = f"{NIH_DIR}/images"

DISEASES = [
    'Atelectasis','Cardiomegaly','Consolidation','Edema',
    'Effusion','Emphysema','Fibrosis','Hernia',
    'Infiltration','Mass','Nodule','Pleural_Thickening',
    'Pneumonia','Pneumothorax'
]

# 1. Load Metadata
# The sample dataset downloaded with this specific CSV name
csv_file = f"{NIH_DIR}/sample_labels.csv"
if not os.path.exists(csv_file):
    # Fallback just in case it was renamed during your move
    csv_file = glob.glob(f"{NIH_DIR}/*.csv")[0]

print(f"Reading metadata from: {csv_file}")
df = pd.read_csv(csv_file)

# 2. Match CSV rows to images physically on Drive
print(f"Validating images in: {IMG_DIR}")
existing_images = set(os.listdir(IMG_DIR))

# Filter dataframe
df_avail = df[df['Image Index'].isin(existing_images)].copy()

# Create absolute path column (Vital for the next training cells)
df_avail['file_path'] = df_avail['Image Index'].apply(lambda x: os.path.join(IMG_DIR, x))

print(f"Total rows in CSV : {len(df):,}")
print(f"Images found on disk : {len(df_avail):,}")

# 3. One-Hot Encoding
for d in DISEASES:
    df_avail[d] = df_avail['Finding Labels'].apply(lambda x: 1 if d in x else 0)
df_avail['is_normal'] = (df_avail['Finding Labels'] == 'No Finding').astype(int)

# 4. Create Stratified Splits (Train: 76.5%, Val: 8.5%, Test: 15%)
# We stratify by 'is_normal' to ensure balanced "Healthy vs Sick" ratios
df_trainval, df_test = train_test_split(
    df_avail, test_size=0.15, random_state=42,
    stratify=df_avail['is_normal']
)

df_train, df_val = train_test_split(
    df_trainval, test_size=0.1, random_state=42,
    stratify=df_trainval['is_normal']
)

# 5. Save Splits to Drive
os.makedirs(DATA_DIR, exist_ok=True)
df_train.to_csv(f"{DATA_DIR}/train.csv", index=False)
df_val.to_csv(f"{DATA_DIR}/val.csv", index=False)
df_test.to_csv(f"{DATA_DIR}/test.csv", index=False)
json.dump(DISEASES, open(f"{DATA_DIR}/diseases.json", 'w'))

print(f"\n Splits saved to {DATA_DIR}:")
print(f"   Train : {len(df_train):,}")
print(f"   Val   : {len(df_val):,}")
print(f"   Test  : {len(df_test):,}")

# 6. Disease Distribution Visualization
counts = df_avail[DISEASES].sum().sort_values()
plt.figure(figsize=(12, 7), facecolor='#08090c')
ax = plt.gca()
ax.set_facecolor('#08090c')

colors = plt.cm.viridis(np.linspace(0.4, 0.8, len(DISEASES)))
counts.plot(kind='barh', color='#22d3a0', alpha=0.9, ax=ax)

plt.title('Pathology Distribution (NIH Sample)', color='white', fontsize=14, pad=20)
plt.xlabel('Count', color='white')
plt.xticks(color='#5a6480')
plt.yticks(color='white')
for sp in ax.spines.values(): sp.set_color('#1e2535')

plt.tight_layout()
os.makedirs(RES_DIR, exist_ok=True)
plt.savefig(f"{RES_DIR}/disease_distribution.png", dpi=150, facecolor='#08090c')
plt.show()

## PyTorch Dataset + DataLoaders

In [ ]:
import torch, cv2, os, json
import numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# --- 1. CONFIGURATION ---
DATA_DIR = "/content/drive/MyDrive/radiology_ai/data"
DISEASES = json.load(open(f"{DATA_DIR}/diseases.json"))

# Medical Image Augmentation Strategy
TRAIN_AUG = A.Compose([
    A.Resize(256, 256),
    A.RandomCrop(224, 224),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.CLAHE(clip_limit=3.0, p=0.4), # Crucial for X-ray contrast
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

VAL_AUG = A.Compose([
    A.Resize(256, 256),
    A.CenterCrop(224, 224),
    A.CLAHE(clip_limit=3.0, p=1.0), # Apply to validation for consistency
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# --- 2. DATASET CLASS ---
class NIHDataset(Dataset):
     def __init__(self, csv_path, transform=None, diseases=DISEASES):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
        self.diseases = diseases

        # Verify the file_path column exists from our EDA step
        if 'file_path' not in self.df.columns:
             # Emergency fallback if Cell 4 wasn't run with absolute paths
            img_base = "/content/drive/MyDrive/radiology_ai/data/nih/images"
            self.df['file_path'] = self.df['Image Index'].apply(lambda x: os.path.join(img_base, x))

        print(f"Dataset loaded from {os.path.basename(csv_path)}: {len(self.df)} images")

     def __len__(self):
        return len(self.df)

     def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['file_path']

        # Load image (BGR to RGB)
        img = cv2.imread(img_path)
        if img is None:
            # Return a zero tensor if image is missing to prevent crash
            return torch.zeros((3, 224, 224)), torch.zeros(len(self.diseases)), row['Image Index']

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

         # Apply Albumentations
        if self.transform:
          img = self.transform(image=img)['image']

         # Convert pathology labels to tensor
        label = torch.tensor([row[d] for d in self.diseases], dtype=torch.float32)

        return img, label, row['Image Index']

# --- 3. CREATE DATALOADERS ---
print("Initializing DataLoaders...")

train_ds = NIHDataset(f"{DATA_DIR}/train.csv", transform=TRAIN_AUG)
val_ds  = NIHDataset(f"{DATA_DIR}/val.csv", transform=VAL_AUG)
test_ds = NIHDataset(f"{DATA_DIR}/test.csv", transform=VAL_AUG)

# Recommended Batch Sizes for T4 GPU
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader  = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

print(f"\n DataLoaders Ready!")
print(f"   Train: {len(train_loader)} batches")
print(f"   Val:   {len(val_loader)} batches")